# Cybershield URL Dataset Pipeline

This notebook builds a **real-URL structural dataset** from Tranco domains using only the **Common Crawl Index API** (no HTML/page crawling).

## Pipeline Steps
1. Load and filter Tranco domains (top N + Arab country TLD focus).
2. Query Common Crawl Index for full URLs per domain.
3. Parse structural URL features.
4. Save dataset and domain-level aggregates.
5. Generate global JSON statistics for synthetic URL generation.

In [53]:

import json
import re
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import parse_qs, urlparse, urlunparse

import pandas as pd
import requests
import joblib

# extracting features

In [ ]:
from __future__ import annotations



# -----------------------------
# Configuration
# -----------------------------

ARAB_TLDS = (".jo")
COMMON_CRAWL_INDEX_API = "https://index.commoncrawl.org"
COMMON_CRAWL_COLLINFO = f"{COMMON_CRAWL_INDEX_API}/collinfo.json"
SPECIAL_CHAR_PATTERN = re.compile(r"[^a-zA-Z0-9/_\-]")
HEX_LIKE_PATTERN = re.compile(r"^[a-f0-9]{16,}$", flags=re.IGNORECASE)
PERCENT_ENCODED_TOKEN_PATTERN = re.compile(r"^(?:%[0-9a-fA-F]{2})+$")
INVALID_QUERY_PUNCT_PATTERN = re.compile(r"[,.()]")
MULTI_SLASH_PATTERN = re.compile(r"/{2,}")


@dataclass
class PipelineConfig:
    tranco_csv_path: str
    output_dir: str = "datasets"
    top_n: int = 300_000
    max_urls_per_domain: int = 50
    max_workers: int = 16
    domain_batch_size: int = 5_000
    request_timeout_sec: int = 20
    max_retries: int = 3
    retry_backoff_sec: float = 1.5
    requests_per_second: float = 12.0


class RateLimiter:
    """Thread-safe fixed-rate limiter shared across all API calls."""

    def __init__(self, calls_per_second: float) -> None:
        if calls_per_second <= 0:
            raise ValueError("calls_per_second must be > 0")
        self.min_interval = 1.0 / calls_per_second
        self._lock = threading.Lock()
        self._last_call_ts = 0.0

    def acquire(self) -> None:
        with self._lock:
            now = time.monotonic()
            wait = self.min_interval - (now - self._last_call_ts)
            if wait > 0:
                time.sleep(wait)
            self._last_call_ts = time.monotonic()


def _detect_domain_column(df: pd.DataFrame) -> str:
    for candidate in ("domain", "Domain", "hostname", "host"):
        if candidate in df.columns:
            return candidate
    if len(df.columns) >= 2:
        return df.columns[1]
    raise ValueError("Could not infer domain column from Tranco file")


def _detect_rank_column(df: pd.DataFrame) -> str:
    for candidate in ("rank", "Rank"):
        if candidate in df.columns:
            return candidate
    if len(df.columns) >= 1:
        return df.columns[0]
    raise ValueError("Could not infer rank column from Tranco file")


def load_tranco(csv_path: str, top_n: int = 300_000) -> pd.DataFrame:
    """Load Tranco CSV and normalize rank/domain columns."""
    csv_path_obj = Path(csv_path)
    if not csv_path_obj.exists():
        raise FileNotFoundError(f"Tranco CSV not found: {csv_path}")

    df = pd.read_csv(csv_path_obj)
    domain_col = _detect_domain_column(df)
    rank_col = _detect_rank_column(df)

    out = df[[rank_col, domain_col]].copy()
    out.columns = ["rank", "domain"]

    out["rank"] = pd.to_numeric(out["rank"], errors="coerce")
    out["domain"] = (
        out["domain"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"^https?://", "", regex=True)
        .str.replace(r"/.*$", "", regex=True)
    )
    out = out.dropna(subset=["rank", "domain"])
    out = out[out["domain"].str.contains(".", regex=False)]
    out = out.sort_values("rank", kind="stable")

    if top_n > 0:
        out = out.head(top_n)
    return out.reset_index(drop=True)


def filter_domains(
    tranco_df: pd.DataFrame,
    top_n: int = 300_000,
    arab_tlds: tuple[str, ...] = ARAB_TLDS,
) -> list[str]:
    """Select top N domains and merge with Arab TLD domains, deduplicated."""
    if tranco_df.empty:
        return []

    working = tranco_df.copy()
    working["rank"] = pd.to_numeric(working["rank"], errors="coerce")
    working = working.dropna(subset=["rank", "domain"]).sort_values("rank", kind="stable")

    top_domains = working.head(top_n)["domain"].tolist()
    arab_domains = working[working["domain"].str.endswith(arab_tlds)]["domain"].tolist()

    merged = top_domains + arab_domains
    deduped = list(dict.fromkeys(merged))
    return deduped


def _request_with_retry(
    session: requests.Session,
    url: str,
    params: dict[str, Any],
    timeout_sec: int,
    max_retries: int,
    backoff_sec: float,
    limiter: RateLimiter,
) -> requests.Response:
    last_error: Exception | None = None
    for attempt in range(max_retries):
        try:
            limiter.acquire()
            response = session.get(url, params=params, timeout=timeout_sec)
            if response.status_code in {429, 500, 502, 503, 504}:
                raise requests.HTTPError(
                    f"Transient HTTP {response.status_code}", response=response
                )
            response.raise_for_status()
            return response
        except (requests.RequestException, requests.HTTPError) as exc:
            last_error = exc
            if attempt < max_retries - 1:
                sleep_for = backoff_sec * (2**attempt)
                time.sleep(sleep_for)
            else:
                break
    raise RuntimeError(f"Request failed after retries: {url} {params}") from last_error


def _get_latest_index_id(
    session: requests.Session,
    limiter: RateLimiter,
    timeout_sec: int,
    max_retries: int,
    backoff_sec: float,
) -> str:
    response = _request_with_retry(
        session=session,
        url=COMMON_CRAWL_COLLINFO,
        params={},
        timeout_sec=timeout_sec,
        max_retries=max_retries,
        backoff_sec=backoff_sec,
        limiter=limiter,
    )
    data = response.json()
    if not data:
        raise RuntimeError("No Common Crawl indexes available")
    # Common Crawl collinfo is ordered newest first; use exactly one latest index.
    return data[0]["id"]


def _extract_urls_from_index_response(text: str) -> list[str]:
    urls: list[str] = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue
        url_value = record.get("url")
        if isinstance(url_value, str) and url_value:
            urls.append(url_value)
    return urls


def is_valid_domain_url(url: str, source_domain: str) -> bool:
    """Allow only exact source domain or subdomains of source domain."""
    try:
        hostname = (urlparse(url).hostname or "").strip(".").lower()
    except ValueError:
        return False

    source = (source_domain or "").strip(".").lower()
    if not hostname or not source:
        return False

    return hostname == source or hostname.endswith(f".{source}")


def clean_token(token: str) -> bool:
    """Filter noisy path tokens used in global token statistics."""
    if token is None:
        return False

    candidate = str(token).strip().lower()
    if len(candidate) < 2:
        return False
    if candidate.isdigit():
        return False
    if HEX_LIKE_PATTERN.match(candidate):
        return False
    if PERCENT_ENCODED_TOKEN_PATTERN.match(candidate) or "%" in candidate:
        return False
    if candidate in {"amp", "nbsp", "quot", "lt", "gt"}:
        return False
    return True


def clean_query_key(key: str) -> bool:
    """Filter malformed/noisy query parameter keys."""
    if key is None:
        return False

    candidate = str(key).strip()
    if not candidate:
        return False
    if len(candidate) > 30:
        return False
    if " " in candidate:
        return False
    if INVALID_QUERY_PUNCT_PATTERN.search(candidate):
        return False
    return True


def _normalize_url_path(url: str) -> str:
    """Normalize only URL path by collapsing multiple slashes."""
    parsed = urlparse(url)
    normalized_path = MULTI_SLASH_PATTERN.sub("/", parsed.path or "")
    normalized = parsed._replace(path=normalized_path)
    return urlunparse(normalized)


def query_commoncrawl(
    domain: str,
    session: requests.Session,
    latest_index_id: str,
    limiter: RateLimiter,
    max_urls_per_domain: int = 50,
    request_timeout_sec: int = 20,
    max_retries: int = 3,
    retry_backoff_sec: float = 1.5,
) -> list[str]:
    """
    Query Common Crawl Index API for full URLs only.
    No page crawl, no HTML fetching.
    Uses only the latest Common Crawl index for speed.
    """
    collected: list[str] = []
    patterns = [f"{domain}/*", f"*.{domain}/*"]

    index_endpoint = f"{COMMON_CRAWL_INDEX_API}/{latest_index_id}-index"
    for pattern in patterns:
        params = {
            "url": pattern,
            "output": "json",
            "fl": "url",
        }
        try:
            response = _request_with_retry(
                session=session,
                url=index_endpoint,
                params=params,
                timeout_sec=request_timeout_sec,
                max_retries=max_retries,
                backoff_sec=retry_backoff_sec,
                limiter=limiter,
            )
        except RuntimeError:
            continue

        extracted = _extract_urls_from_index_response(response.text)
        if extracted:
            for candidate_url in extracted:
                if not is_valid_domain_url(candidate_url, domain):
                    continue
                collected.append(candidate_url)

        if len(collected) >= max_urls_per_domain:
            break

    deduped = list(dict.fromkeys(collected))
    return deduped[:max_urls_per_domain]


def _heuristic_domain_parts(hostname: str) -> tuple[str, str, str]:
    """Return (registrable_domain, tld, subdomain) via lightweight heuristic."""
    if not hostname:
        return "", "", ""

    parts = hostname.split(".")
    if len(parts) < 2:
        return hostname, "", ""

    tld = parts[-1]
    registrable = ".".join(parts[-2:])
    subdomain = ".".join(parts[:-2]) if len(parts) > 2 else ""
    return registrable, tld, subdomain


def parse_url(url: str) -> dict[str, Any]:
    """Extract structural URL components (no ML features)."""
    normalized_url = _normalize_url_path(url)
    parsed = urlparse(normalized_url)
    hostname = (parsed.hostname or "").lower()

    registrable_domain, tld, heuristic_subdomain = _heuristic_domain_parts(hostname)
    path = parsed.path or ""
    path_segments = [segment for segment in path.split("/") if segment]

    query_map = parse_qs(parsed.query, keep_blank_values=True)
    raw_query_keys = list(query_map.keys())
    query_keys = [k for k in raw_query_keys if clean_query_key(k)]

    has_digits = bool(re.search(r"\d", path))
    has_special_chars = bool(SPECIAL_CHAR_PATTERN.search(path))

    return {
        "full_url": normalized_url,
        "parsed_domain": registrable_domain,
        "tld": tld,
        "subdomain": heuristic_subdomain,
        "path": path,
        "path_segments": path_segments,
        "query_keys": query_keys,
        "query_count": len(query_keys),
        "path_depth": len(path_segments),
        "url_length": len(normalized_url),
        "num_path_segments": len(path_segments),
        "has_digits": has_digits,
        "has_special_chars": has_special_chars,
    }


def _infer_subdomain_relative_to_source(hostname: str, source_domain: str) -> str:
    hostname = (hostname or "").lower()
    source_domain = (source_domain or "").lower()
    if not hostname or not source_domain:
        return ""
    if hostname == source_domain:
        return ""
    if hostname.endswith(f".{source_domain}"):
        return hostname[: -(len(source_domain) + 1)]
    return ""


def _count_subdomain_levels(subdomain: str) -> int:
    if not subdomain:
        return 0
    return len([part for part in subdomain.split(".") if part])


def _domain_stats(dataset_df: pd.DataFrame) -> pd.DataFrame:
    if dataset_df.empty:
        return pd.DataFrame(
            columns=[
                "domain",
                "url_count",
                "avg_path_depth",
                "avg_query_count",
                "avg_url_length",
                "avg_subdomain_levels",
                "pct_has_digits",
                "pct_has_special_chars",
            ]
        )

    grouped = dataset_df.groupby("domain", as_index=False).agg(
        url_count=("full_url", "count"),
        avg_path_depth=("path_depth", "mean"),
        avg_query_count=("query_count", "mean"),
        avg_url_length=("url_length", "mean"),
        avg_subdomain_levels=("num_subdomain_levels", "mean"),
        pct_has_digits=("has_digits", "mean"),
        pct_has_special_chars=("has_special_chars", "mean"),
    )
    return grouped


def _global_stats(dataset_df: pd.DataFrame) -> dict[str, Any]:
    if dataset_df.empty:
        return {
            "most_common_path_tokens": {},
            "path_depth_distribution": {},
            "query_parameter_frequency": {},
            "subdomain_frequency": {},
            "url_length_distribution": {},
        }

    path_token_counter = Counter()
    query_key_counter = Counter()

    for row in dataset_df.itertuples(index=False):
        for segment in row.path_segments:
            segment_norm = str(segment).strip().lower()
            if clean_token(segment_norm):
                path_token_counter[segment_norm] += 1

        for key in row.query_keys:
            key_norm = str(key).strip().lower()
            if clean_query_key(key_norm):
                query_key_counter[key_norm] += 1

    path_depth_distribution = (
        dataset_df["path_depth"].value_counts().sort_index().astype(int).to_dict()
    )
    subdomain_frequency = (
        dataset_df["subdomain"]
        .fillna("")
        .replace("", "<root>")
        .value_counts()
        .head(200)
        .astype(int)
        .to_dict()
    )

    length_bins = [0, 50, 75, 100, 125, 150, 200, 300, 500, 1000, 10_000]
    length_labels = [
        "0-50",
        "51-75",
        "76-100",
        "101-125",
        "126-150",
        "151-200",
        "201-300",
        "301-500",
        "501-1000",
        "1001+",
    ]
    bucketed = pd.cut(
        dataset_df["url_length"],
        bins=length_bins,
        labels=length_labels,
        include_lowest=True,
        right=True,
    )
    url_length_distribution = bucketed.value_counts().sort_index().astype(int).to_dict()

    return {
        "most_common_path_tokens": dict(path_token_counter.most_common(100)),
        "path_depth_distribution": path_depth_distribution,
        "query_parameter_frequency": dict(query_key_counter.most_common(100)),
        "subdomain_frequency": subdomain_frequency,
        "url_length_distribution": {
            str(k): int(v) for k, v in url_length_distribution.items()
        },
    }


def build_dataset(
    domains: list[str],
    max_urls_per_domain: int = 50,
    max_workers: int = 16,
    domain_batch_size: int = 5_000,
    request_timeout_sec: int = 20,
    max_retries: int = 3,
    retry_backoff_sec: float = 1.5,
    requests_per_second: float = 12.0,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    """Build URL structure dataset from domains using parallel Common Crawl index queries."""
    if not domains:
        empty_dataset = pd.DataFrame(
            columns=[
                "domain",
                "full_url",
                "tld",
                "subdomain",
                "num_subdomain_levels",
                "path",
                "path_depth",
                "query_count",
                "url_length",
                "num_path_segments",
                "has_digits",
                "has_special_chars",
            ]
        )
        return empty_dataset, _domain_stats(empty_dataset), _global_stats(empty_dataset)

    limiter = RateLimiter(requests_per_second)

    with requests.Session() as session:
        latest_index_id = _get_latest_index_id(
            session=session,
            limiter=limiter,
            timeout_sec=request_timeout_sec,
            max_retries=max_retries,
            backoff_sec=retry_backoff_sec,
        )
        print(f"Using Common Crawl index: {latest_index_id}")

        all_rows: list[dict[str, Any]] = []
        total_domains = len(domains)

        for batch_start in range(0, total_domains, domain_batch_size):
            batch_end = min(batch_start + domain_batch_size, total_domains)
            domain_batch = domains[batch_start:batch_end]
            print(f"Processing domains {batch_start + 1}-{batch_end} / {total_domains}")

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                future_map = {
                    executor.submit(
                        query_commoncrawl,
                        domain,
                        session,
                        latest_index_id,
                        limiter,
                        max_urls_per_domain,
                        request_timeout_sec,
                        max_retries,
                        retry_backoff_sec,
                    ): domain
                    for domain in domain_batch
                }

                for future in as_completed(future_map):
                    source_domain = future_map[future]
                    try:
                        urls = future.result()
                    except Exception as exc:  # noqa: BLE001
                        print(f"[WARN] Domain failed: {source_domain} ({exc})")
                        continue

                    for url in urls:
                        if not is_valid_domain_url(url, source_domain):
                            continue

                        parsed = parse_url(url)
                        hostname = (urlparse(parsed["full_url"]).hostname or "").lower()
                        subdomain_relative = _infer_subdomain_relative_to_source(
                            hostname, source_domain
                        )
                        final_subdomain = subdomain_relative or parsed["subdomain"]

                        row = {
                            "domain": source_domain,
                            "full_url": parsed["full_url"],
                            "tld": parsed["tld"],
                            "subdomain": final_subdomain,
                            "num_subdomain_levels": _count_subdomain_levels(final_subdomain),
                            "path": parsed["path"],
                            "path_depth": parsed["path_depth"],
                            "query_count": parsed["query_count"],
                            "url_length": parsed["url_length"],
                            "num_path_segments": parsed["num_path_segments"],
                            "has_digits": parsed["has_digits"],
                            "has_special_chars": parsed["has_special_chars"],
                            "path_segments": parsed["path_segments"],
                            "query_keys": parsed["query_keys"],
                        }
                        all_rows.append(row)

    dataset_df = pd.DataFrame(all_rows)
    if not dataset_df.empty:
        dataset_df = dataset_df.drop_duplicates(subset=["domain", "full_url"]).reset_index(
            drop=True
        )

    domain_stats_df = _domain_stats(dataset_df)
    global_stats = _global_stats(dataset_df)

    # Keep heavy list columns only for in-memory stats, not in final persisted dataset.
    dataset_df = dataset_df.drop(columns=["path_segments", "query_keys"], errors="ignore")

    return dataset_df, domain_stats_df, global_stats


def save_outputs(
    dataset_df: pd.DataFrame,
    domain_stats_df: pd.DataFrame,
    global_stats: dict[str, Any],
    output_dir: str,
    base_name: str = "tranco_commoncrawl_url_structures",
) -> dict[str, str]:
    """Persist outputs to parquet/csv + JSON."""
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    dataset_parquet = out_dir / f"{base_name}.parquet"
    dataset_csv = out_dir / f"{base_name}.csv"
    domain_stats_csv = out_dir / f"{base_name}_domain_stats.csv"
    global_stats_json = out_dir / f"{base_name}_global_stats.json"

    paths_written: dict[str, str] = {}

    try:
        dataset_df.to_parquet(dataset_parquet, index=False)
        paths_written["dataset_parquet"] = str(dataset_parquet)
    except Exception:  # noqa: BLE001
        dataset_df.to_csv(dataset_csv, index=False)
        paths_written["dataset_csv_fallback"] = str(dataset_csv)

    domain_stats_df.to_csv(domain_stats_csv, index=False)
    paths_written["domain_stats_csv"] = str(domain_stats_csv)

    with global_stats_json.open("w", encoding="utf-8") as f:
        json.dump(global_stats, f, indent=2, ensure_ascii=True)
    paths_written["global_stats_json"] = str(global_stats_json)

    return paths_written

In [21]:
# get tranco csv top 100 domains plus arab tlds
df = pd.read_csv("datasets/archive/tranco_L7P34.csv")
df = df.rename(columns={df.columns[0]: "rank", df.columns[1]: "domain"})

In [22]:
filtered_domains = filter_domains(df, top_n=100, arab_tlds=ARAB_TLDS)

In [ ]:
filtered_domains_df = pd.DataFrame({"domain": filtered_domains})

In [24]:
# Example execution (adjust paths and scale settings as needed)
config = PipelineConfig(
    tranco_csv_path="datasets/archive/tranco_L7P34.csv",  # update if needed
    output_dir="datasets",
    top_n=10,
    max_urls_per_domain=30,
    max_workers=12,
    domain_batch_size=2_000,
    requests_per_second=10.0,
)

tranco_df = load_tranco(config.tranco_csv_path, top_n=config.top_n)
domains = filtered_domains
print(f"Selected domains: {len(domains):,}")

dataset_df, domain_stats_df, global_stats = build_dataset(
    domains=domains,
    max_urls_per_domain=config.max_urls_per_domain,
    max_workers=config.max_workers,
    domain_batch_size=config.domain_batch_size,
    request_timeout_sec=config.request_timeout_sec,
    max_retries=config.max_retries,
    retry_backoff_sec=config.retry_backoff_sec,
    requests_per_second=config.requests_per_second,
)

written = save_outputs(
    dataset_df=dataset_df,
    domain_stats_df=domain_stats_df,
    global_stats=global_stats,
    output_dir=config.output_dir,
)

print("Rows in dataset:", len(dataset_df))
print("Rows in domain stats:", len(domain_stats_df))
print("Output files:")
for key, value in written.items():
    print(f"- {key}: {value}")

dataset_df.head(10)

Selected domains: 250
Using Common Crawl index: CC-MAIN-2026-12
Processing domains 1-250 / 250
Rows in dataset: 526
Rows in domain stats: 26
Output files:
- dataset_csv_fallback: datasets\tranco_commoncrawl_url_structures.csv
- domain_stats_csv: datasets\tranco_commoncrawl_url_structures_domain_stats.csv
- global_stats_json: datasets\tranco_commoncrawl_url_structures_global_stats.json


,domain,full_url,tld,subdomain,num_subdomain_levels,path,path_depth,query_count,url_length,num_path_segments,has_digits,has_special_chars
0,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.VmXeeQCElfk.O/am=g...,9,0,969,9,True,True
1,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.VmXeeQCElfk.O/am=g...,9,0,237,9,True,True
2,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.VmXeeQCElfk.O/am=g...,9,0,155,9,True,True
3,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.VmXeeQCElfk.O/am=g...,9,0,136,9,True,True
4,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.VmXeeQCElfk.O/am=g...,9,0,273,9,True,True
5,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.VmXeeQCElfk.O/am=g...,9,0,127,9,True,True
6,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.VmXeeQCElfk.O/am=g...,9,0,120,9,True,True
7,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.xJZxFwk0ZHI.O/d=0/...,8,0,226,8,True,True
8,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.xJZxFwk0ZHI.O/d=0/...,8,0,871,8,True,True
9,gstatic.com,https://www.gstatic.com/_/atari/_/js/k=atari.v...,com,www,1,/_/atari/_/js/k=atari.vw.de.xJZxFwk0ZHI.O/d=0/...,8,0,153,8,True,True


# Synthesising full URLs

In [25]:
import pandas as pd
import random
from urllib.parse import urlunparse, urlencode
from typing import Dict, Any, List


class LegitURLDatasetFromJSON:
    def __init__(self, global_stats: Dict[str, Any]):
        self.path_tokens = self._to_weighted(global_stats["most_common_path_tokens"])
        self.query_keys = self._to_weighted(global_stats["query_parameter_frequency"])
        self.path_depths = self._to_weighted(global_stats["path_depth_distribution"])
        self.subdomains = self._to_weighted(global_stats["subdomain_frequency"])

    # -----------------------------
    # convert frequency dict → weighted sampling list
    # -----------------------------

    def _to_weighted(self, freq: Dict[str, int]) -> List[str]:
        items = []
        for k, v in freq.items():
            items.extend([k] * max(1, int(v)))
        return items or [""]

    # -----------------------------
    # sampling helpers
    # -----------------------------

    def _sample_domain(self, domains: List[str]) -> str:
        return random.choice(domains)

    def _sample_subdomain(self) -> str:
        sub = random.choice(self.subdomains)
        return "" if sub == "<root>" else sub

    def _sample_depth(self) -> int:
        return int(random.choice(self.path_depths))

    def _sample_path(self, depth: int) -> str:
        tokens = [random.choice(self.path_tokens) for _ in range(depth)]
        return "/" + "/".join(tokens)

    def _sample_query(self) -> dict:
        q = {}
        if random.random() < 0.4:
            for _ in range(random.randint(1, 2)):
                key = random.choice(self.query_keys)
                q[key] = str(random.randint(1000, 999999))
        return q

    # -----------------------------
    # main generator
    # -----------------------------

    def generate_dataframe(
        self,
        domains: List[str],
        num_samples: int,
    ) -> pd.DataFrame:

        rows = []

        for _ in range(num_samples):
            domain = self._sample_domain(domains)
            sub = self._sample_subdomain()
            depth = self._sample_depth()

            path = self._sample_path(depth)
            query = self._sample_query()

            netloc = f"{sub}.{domain}" if sub else domain
            query_str = urlencode(query)

            url = urlunparse(
                ("https", netloc, path, "", query_str, "")
            )

            rows.append(
                {
                    "url": url,
                    "domain": domain,
                    "label": 0,  # legit
                    "path_depth": depth,
                    "num_query_params": len(query),
                    "subdomain": sub,
                    "url_length": len(url),
                }
            )

        return pd.DataFrame(rows)

In [28]:
import os
print(os.getcwd())

c:\Users\Lenovo\Documents\02 Programming\CPF Government app\project\backend\app\ml


In [92]:
domains = pd.read_csv("app/ml/datasets/archive/tranco_L7P34.csv")
domains = domains[:300000]
domains = domains[domains.columns[1]]

In [93]:
domains.head()

0    gtld-servers.net
1         gstatic.com
2        facebook.com
3      cloudflare.com
4       microsoft.com
Name: google.com, dtype: object

In [96]:
import json

# load your actual output
with open("app/ml/datasets/archive/tranco_commoncrawl_url_structures_global_stats.json", "r") as f:
    global_stats = json.load(f)


generator = LegitURLDatasetFromJSON(global_stats)

df = generator.generate_dataframe(
    domains=domains,
    num_samples=800000
)

print(df.head())

                                                 url             domain  \
0    https://www.softwar.io/ar/docidredir.aspx/atari         softwar.io   
1  https://jaf.linksynergy.com/?id=418320&newsid=...    linksynergy.com   
2  https://static.clientportal.link/ar/uploadscv/...  clientportal.link   
3   https://www.role-me.ru/faculty-arts-and-sciences         role-me.ru   
4                             https://www.kaya.in/v1            kaya.in   

   label  path_depth  num_query_params subdomain  url_length  
0      0           3                 0       www          47  
1      0           0                 2       jaf          52  
2      0           5                 0    static          78  
3      0           1                 0       www          48  
4      0           1                 0       www          22  


In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   url               800000 non-null  object
 1   domain            800000 non-null  object
 2   label             800000 non-null  int64 
 3   path_depth        800000 non-null  int64 
 4   num_query_params  800000 non-null  int64 
 5   subdomain         800000 non-null  object
 6   url_length        800000 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 42.7+ MB


In [99]:
df.to_csv("app/ml/datasets/synthetic_legit_urls.csv", index=False)

# build dataset

In [56]:
print(os.getcwd())

c:\Users\Lenovo\Documents\02 Programming\CPF Government app\project\backend\app\ml


In [57]:
print(os.getcwd())
os.chdir("../..")
print(os.getcwd())

c:\Users\Lenovo\Documents\02 Programming\CPF Government app\project\backend\app\ml
c:\Users\Lenovo\Documents\02 Programming\CPF Government app\project\backend


In [58]:
import app.ml.Preprocessing

In [72]:
from huggingface_hub import hf_hub_download
MODEL_PATH = hf_hub_download(
    repo_id="aboudiua/url_phishing_model_pipeline",
    repo_type="model",
    filename="model.joblib"
)
loaded_model = joblib.load(MODEL_PATH)

In [114]:
loaded_model = joblib.load("app/ml/models/url_phishing_model_pipeline1.joblib")

In [115]:
res = loaded_model.predict_proba(["https://eshop.umniah.com/en/fiber-790.html"])
print(res)

[[0.4 0.6]]


In [ ]:
import pandas as pd

file_path = "app/ml/datasets/synthetic_legit_urls.csv"

positive_urls = []
target_hits = 800_000
chunk_size = 10_000
total_checks = 0
for chunk in pd.read_csv(file_path, usecols=["url"], chunksize=chunk_size):

    urls = chunk["url"].astype(str).tolist()

    # predict probabilities in batch
    probs = loaded_model.predict_proba(urls)[:, 1]  # class 1 probability

    preds = (probs >= 0.5).astype(int)  # adjust threshold if needed

    mask = preds == 1

    positive_urls.extend([u for u, m in zip(urls, mask) if m])

    if len(positive_urls) >= target_hits:
        positive_urls = positive_urls[:target_hits]
        break
    total_checks += 1*chunk_size
print("Collected:", len(positive_urls))
print("total:", total_checks)

Collected: 32
total: 800000
ratio:  25000.0


In [117]:
len(positive_urls)

32

In [118]:
df = pd.DataFrame(positive_urls)

In [120]:
df.head(10)

,0
0,https://lycee-charlesdegaulle.fr/
1,https://instituteforapprenticeships.org/
2,https://xn--vhqz73ae0bee03ut4hpvbb23hew3b.com/
3,https://www.netlify.app/
4,https://shhhnewcastleswingers.club/
5,https://melbetofficialbookmaker.ru/
6,https://cardrush-vanguard.jp/
7,https://einzahlungbonus.de/
8,https://ioioioioioio.com/?view=4013&q=492253
9,https://ttjmemnvpdsmcfvnmdkvmsdfvlefnslsdeakfn...


In [113]:
df.to_csv("app/ml/datasets/archive/synthetic_legit_urls_26k_iteration2.csv")